> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.9 - Visualização de Dados

## Exercícios
Vamos trabalhar com as mesmas bases de dados do exercício de Pandas. Aqui estão os links caso você queira baixar novamente, mas recomendo trabalhar com o `wc_formatado.csv` que exportamos na questão Q2 do exercício anterior.

* https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/fifa-wc/matches_1930_2022.csv
* https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/fifa-wc/matches_1991_2023.csv

Para relembrar, essas são as colunas do dataframe:
```
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   time_1             1312 non-null   string        
 1   time_2             1312 non-null   string        
 2   gols_1             1312 non-null   int64         
 3   gols_2             1312 non-null   int64         
 4   data               1312 non-null   datetime64[ns]
 5   ano                1312 non-null   int64         
 6   país_sede          1312 non-null   string        
 7   comparecimento     1312 non-null   int64         
 8   resultado          1312 non-null   string        
 9   rodada             1312 non-null   category      
 10  gols_1_detalhes    970 non-null    string        
 11  gols_2_detalhes    771 non-null    string        
 12  gols_1_contra      57 non-null     string        
 13  gols_2_contra      30 non-null     string        
 14  gols_1_penalti     170 non-null    string        
 15  gols_2_penalti     119 non-null    string        
 16  cartao_vermelho_1  59 non-null     string        
 17  cartao_vermelho_2  65 non-null     string        
 18  cartao_amarelo_1   834 non-null    string        
 19  cartao_amarelo_2   857 non-null    string        
 20  copa               1312 non-null   string 
```

#### Q1.
Realize todos os imports necessários para executar as três bibliotecas de visualização que conhecemos:
* Matplotlib (lembre-se do comando mágico)
* Seaborn
* Plotly

Para cada uma delas, altere o tema padrão de visualização. 

In [ ]:
import pandas as pd
import csv
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',12)

wcwomen_df = pd.read_csv('matches_1991_2023.csv')
wcmen_df   = pd.read_csv('matches_1930_2022.csv')
wc = pd.concat((wcwomen_df,wcmen_df)).reset_index()

nomes_traduzidos = {'home_team': 'time_1', 'away_team': 'time_2', 'home_score': 'gols_1', 'away_score': 'gols_2',
                    'Date': 'data', 'Year': 'ano', 'Host': 'país_sede', 'Attendance': 'comparecimento',
                    'Score': 'resultado', 'Round': 'rodada', 'home_goal': 'gols_1_detalhes', 'away_goal': 'gols_2_detalhes',
                    'home_own_goal': 'gols_1_contra', 'away_own_goal': 'gols_2_contra',
                    'home_penalty_goal': 'gols_1_penalti', 'away_penalty_goal': 'gols_2_penalti',
                    'home_red_card': 'cartao_vermelho_1', 'away_red_card': 'cartao_vermelho_2',
                    'home_yellow_card_long': 'cartao_amarelo_1', 'away_yellow_card_long': 'cartao_amarelo_2'}

wc = wc.loc[:, nomes_traduzidos.keys()]
wc.columns = nomes_traduzidos.values()

copa = wc['ano'].apply( lambda x: 'Masculina' if x % 2 == 0 else 'Feminina').astype('string')
wc['copa'] = copa


In [ ]:
import numpy as np
import  matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import plotly.express as px
%matplotlib inline

plt.style.use('ggplot')
sns.set_theme(style='darkgrid', palette='muted')
pio.templates.default = "plotly_dark"

#### Q2.
Sobre os dados de copa do mundo, qual a distribuição de público presente nos jogos? Isso pode ser respondido com um histograma com os dados da coluna `comparecimento`.  

Lembre-se que alguns jogos estavam com público 0 incorretamente, que tal remover essas ocorrências para não atrapalhar sua visualzação?

Você deve implementar essa visualização nas três bibliotecas que vimos:
* Matplotlib
* Seaborn
* Plotly

Garanta que o gráfico tenha pelo menos os atributos de título e rótulos de dimensão.

In [ ]:
#### Solução com matplotlib
wc_filtrado = wc[wc['comparecimento'] > 0].dropna(subset=['comparecimento'])

plt.figure(figsize=(9, 5))

plt.hist(wc_filtrado['comparecimento'], bins=20, color='royalblue')

plt.title('Distribuição de Comparecimento nas Partidas da Copa do Mundo', fontsize=12)
plt.xlabel('Comparecimento (Número de Espectadores)', fontsize=10)
plt.ylabel('Quantidade de Partidas', fontsize=10)

In [ ]:
#### solução com seaborn
plt.figure(figsize=(9, 5))

sns.histplot(data=wc_filtrado, x='comparecimento', bins=20, color='teal')

plt.title('Distribuição de Comparecimento nas Partidas da Copa do Mundo', fontsize=12)
plt.xlabel('Comparecimento (Número de Espectadores)', fontsize=10)
plt.ylabel('Quantidade de Partidas', fontsize=10)

plt.show()

In [ ]:
#### solução com plotly
import plotly.express as px
pio.renderers.default = 'notebook'

fig = px.histogram(
    wc_filtrado, 
    x='comparecimento', 
    nbins=20,
    title='Distribuição de Comparecimento nas Partidas da Copa do Mundo',
    labels={
        'comparecimento': 'Comparecimento (Número de Espectadores)',
        'count': 'Quantidade de Partidas'
    }
)

fig.update_traces(marker_line_color='black', marker_line_width=1)

fig.show()

#### Q3.

Apresente um gráfico de dispersão (scatter) dos atributos `gols_1` e `gols_2`. Isso representa a relação entre gols feitos e gols tomados por jogo. Há alguma relação interessante entre esses atributos?

Para facilitar a visualização dos dados (já que tem muitos placares repetidos), aplique uma leve distorção aos dados para que cada ponto esteja deslocado aleatoriamente de seu valor original. Código apresentado a seguir
```python
gols = wc[['gols_1', 'gols_2']] * np.random.random((len(wc),2))
```

Você deve implementar essa visualização nas três bibliotecas que vimos:
* Matplotlib
* Seaborn
* Plotly

Garanta que o gráfico tenha pelo menos os atributos de título e rótulos de dimensão.

In [ ]:
#### solução com matplotlib
import numpy as np
gols = wc[['gols_1', 'gols_2']] * np.random.random((len(wc),2))
plt.figure(figsize=(8, 6))

plt.scatter(gols['gols_1'], gols['gols_2'], alpha=0.5, color='crimson')

plt.title('Relação entre Gols Feitos (Time 1) e Gols Tomados (Time 2) por Jogo', fontsize=12)
plt.xlabel('Gols do Time 1', fontsize=10)
plt.ylabel('Gols do Time 2', fontsize=10)

plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
#### solução com seaborn
plt.figure(figsize=(8, 6))

sns.scatterplot(data=gols, x='gols_1', y='gols_2', alpha=0.6, color='teal')

plt.title('Relação entre Gols Feitos (Time 1) e Gols Tomados (Time 2) por Jogo', fontsize=12)
plt.xlabel('Gols do Time 1', fontsize=10)
plt.ylabel('Gols do Time 2', fontsize=10)

plt.show()

In [ ]:
#### solução com plotly
import plotly.express as px
pio.renderers.default = 'notebook'

fig = px.scatter(
    gols, 
    x='gols_1', 
    y='gols_2',
    opacity=0.6,
    title='Relação entre Gols Feitos (Time 1) e Gols Tomados (Time 2) por Jogo',
    labels={
        'gols_1': 'Gols do Time 1',
        'gols_2': 'Gols do Time 2'
    }
)

# Ajuste visual nos marcadores
fig.update_traces(marker=dict(size=7, color='darkblue'))

fig.show()

#### Q4.

Apresente um gráfico de barras com o top 10 países que mais participaram de copas do mundo, onde no eixo x devem estar o nome dos países e no eixo y a contagem de participações. Você deve separar a contagem de participações em copas femininas e masculinas, empilhando as barras de cada informação.

No exemplo de barras empilhadas da galeria do matplotlib, imagine que a parte azul são as participações do país em copas masculinas, e em laranja as participações femininas:
* https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_stacked.html

Você deve implementar essa visualização nas três bibliotecas que vimos:
* Matplotlib
* Seaborn
* Plotly

Garanta que o gráfico tenha pelo menos os atributos:
* título
* rótulos de dimensão.
* legenda

In [ ]:
t1 = wc[['time_1', 'ano', 'copa']].rename(columns={'time_1': 'pais'})
t2 = wc[['time_2', 'ano', 'copa']].rename(columns={'time_2': 'pais'})
paises_jogos = pd.concat([t1, t2])

# 2. Contar edições (anos) ÚNICAS que cada país jogou por modalidade
participacoes = (
    paises_jogos.groupby(['pais', 'copa'])['ano'].nunique().reset_index()
)

# 3. Pivotar para ter colunas 'Masculina' e 'Feminina' por linha de país
top10 = participacoes.pivot(
    index='pais', columns='copa', values='ano'
).fillna(0)

# 4. Criar o Total, ordenar e pegar o Top 10
top10['total'] = top10['Masculina'] + top10['Feminina']
top10 = top10.sort_values(by='total', ascending=False).head(10).reset_index()

In [ ]:
#### solução com matplotlib
plt.figure(figsize=(10, 6))

plt.bar(
    top10['pais'],
    top10['Masculina'],
    label='Copa Masculina',
    color='royalblue',
)

plt.bar(
    top10['pais'],
    top10['Feminina'],
    bottom=top10['Masculina'],
    label='Copa Feminina',
    color='darkorange',
)

plt.title(
    'Top 10 Países com Mais Participações em Copas do Mundo', fontsize=13
)
plt.xlabel('País', fontsize=11)
plt.ylabel('Contagem de Participações (Edições)', fontsize=11)
plt.xticks(rotation=45)
plt.legend(title='Modalidade')

plt.tight_layout()
plt.show()

In [ ]:
#### solução com seaborn
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top10,
    x='pais',
    y='total',
    color='darkorange',
    label='Copa Feminina',
)

sns.barplot(
    data=top10,
    x='pais',
    y='Masculina',
    color='royalblue',
    label='Copa Masculina',
)

plt.title(
    'Top 10 Países com Mais Participações em Copas do Mundo', fontsize=13
)
plt.xlabel('País', fontsize=11)
plt.ylabel('Contagem de Participações (Edições)', fontsize=11)
plt.xticks(rotation=45)
plt.legend(title='Modalidade')

plt.tight_layout()
plt.show()

In [ ]:
#### solução com plotly
top10_melted = top10.melt(
    id_vars=['pais'],
    value_vars=['Masculina', 'Feminina'],
    var_name='Modalidade',
    value_name='Participacoes',
)

fig = px.bar(
    top10_melted,
    x='pais',
    y='Participacoes',
    color='Modalidade',
    title='Top 10 Países com Mais Participações em Copas do Mundo',
    labels={
        'pais': 'País',
        'Participacoes': 'Contagem de Participações (Edições)',
    },
    color_discrete_map={
        'Masculina': 'royalblue',
        'Feminina': 'darkorange',
    },
)

fig.show()

#### Q5.

Vamos fazer um compilado com as estatísticas históricas de copas do mundo!

Com a biblioteca de sua preferência você deve criar 4 subplots organizados em um grid de 2 linhas e 2 colunas. Eles devem conter os seguintes gráficos:
* Linha 1, coluna 1: Gráfico de barras com a quantidade de jogos que aconteceram por ano
* Linha 1, coluna 2: Gráfico de área (referências a seguir) com o total de gols por ano, separando as informações de `gols_1` e `gols_2` para distinguir gols em casa e do time visitante.
* Linha 2, coluna 1: Gráfico de área com o total de cartões por ano, separando as informações de cartões amarelos e cartões vermelhos, mas agregando cartões do time 1 ou time 2. Ou seja, uma área com `cartao_amarelo_1 + cartao_amarelo_2` e outra área com `cartao_vermelho_1 + cartao_vermelho_2`.
* Linha 2, coluna 2: Gráfico de barras com o total de gols contra por ano, somando `gols_contra_1` e `gols_contra_2`.

Referências sobre gráfico de área
* Matplotlib: https://matplotlib.org/stable/gallery/lines_bars_and_markers/stackplot_demo.html#sphx-glr-gallery-lines-bars-and-markers-stackplot-demo-py
* Pandas + Matplotlib: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.area.html
* Seaborn: https://seaborn.pydata.org/generated/seaborn.objects.Area.html
* Plotly: https://plotly.com/python/filled-area-plots/

In [ ]:
def contar_eventos(val):
    if pd.isna(val) or val == '' or val == 0:
        return 0
    if isinstance(val, (int, float)):
        return int(val)
    
    val_str = str(val).strip()
    if not val_str or val_str.lower() in ['nan', 'none', '0']:
        return 0
    if '|' in val_str:
        return len(val_str.split('|'))
    if ',' in val_str:
        return len(val_str.split(','))

    return 1
    
colunas_cartoes = [
    'cartao_amarelo_1',
    'cartao_amarelo_2',
    'cartao_vermelho_1',
    'cartao_vermelho_2',
]
for col in colunas_cartoes:
    if col in wc.columns:
        wc[col] = wc[col].apply(contar_eventos)

jogos_por_ano = wc.groupby('ano').size()

wc['gols_1'] = pd.to_numeric(wc['gols_1'], errors='coerce').fillna(0)
wc['gols_2'] = pd.to_numeric(wc['gols_2'], errors='coerce').fillna(0)
gols_por_ano = wc.groupby('ano')[['gols_1', 'gols_2']].sum()

wc['cartao_amarelo_total'] = wc['cartao_amarelo_1'] + wc['cartao_amarelo_2']
wc['cartao_vermelho_total'] = wc['cartao_vermelho_1'] + wc['cartao_vermelho_2']

cartoes_por_ano = wc.groupby('ano')[
    ['cartao_amarelo_total', 'cartao_vermelho_total']
].sum()

wc['gols_contra_total'] = wc['gols_1_contra'] + wc['gols_2_contra']
gols_contra_por_ano = wc.groupby('ano')['gols_contra_total'].sum()

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
fig.suptitle(
    'Compilado de Estatísticas Históricas da Copa do Mundo',
    fontsize=16,
    fontweight='bold',
    y=0.98,
)

# L1C1: Jogos por ano
axes[0, 0].bar(
    jogos_por_ano.index,
    jogos_por_ano.values,
    color='royalblue',
    edgecolor='black',
    alpha=0.8,
)
axes[0, 0].set_title('Quantidade de Jogos por Ano', fontweight='bold')
axes[0, 0].set_xlabel('Ano')
axes[0, 0].set_ylabel('Nº de Jogos')
axes[0, 0].grid(axis='y', linestyle='--', alpha=0.5)

# L1C2: Gols por ano
axes[0, 1].stackplot(
    gols_por_ano.index,
    gols_por_ano['gols_1'],
    gols_por_ano['gols_2'],
    labels=['Gols Mandante', 'Gols Visitante'],
    colors=['#2ca02c', '#d62728'],
    alpha=0.7,
)
axes[0, 1].set_title(
    'Total de Gols por Ano (Mandante vs Visitante)', fontweight='bold'
)
axes[0, 1].set_xlabel('Ano')
axes[0, 1].set_ylabel('Total de Gols')
axes[0, 1].legend(loc='upper left')
axes[0, 1].grid(axis='y', linestyle='--', alpha=0.5)

# L2C1: Cartões por ano
axes[1, 0].stackplot(
    cartoes_por_ano.index,
    cartoes_por_ano['cartao_amarelo_total'],
    cartoes_por_ano['cartao_vermelho_total'],
    labels=['Cartões Amarelos', 'Cartões Vermelhos'],
    colors=['#ff7f0e', '#d62728'],
    alpha=0.7,
)
axes[1, 0].set_title(
    'Total de Cartões por Ano (Amarelos vs Vermelhos)', fontweight='bold'
)
axes[1, 0].set_xlabel('Ano')
axes[1, 0].set_ylabel('Total de Cartões')
axes[1, 0].legend(loc='upper left')
axes[1, 0].grid(axis='y', linestyle='--', alpha=0.5)

# L2C2: Gols Contra por ano
axes[1, 1].bar(
    gols_contra_por_ano.index,
    gols_contra_por_ano.values,
    color='darkred',
    edgecolor='black',
    alpha=0.8,
)
axes[1, 1].set_title('Total de Gols Contra por Ano', fontweight='bold')
axes[1, 1].set_xlabel('Ano')
axes[1, 1].set_ylabel('Gols Contra')
axes[1, 1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
